# Wildfire GRPO â€” End-to-End Training Notebook

Complete pipeline for the Wildfire OpenEnv hackathon submission. Phases:

1. **Install** â€” clone the repo and install training extras
2. **Preflight** â€” validate the OpenEnv manifest and run a 1-iter smoke test
3. **Untrained baseline** â€” evaluate Qwen3-4B zero-shot (no LoRA) on held-out seeds
4. **Training** â€” multi-turn GRPO over the 3-tier curriculum
5. **Trained eval** â€” re-run the same held-out seeds with the trained adapter
6. **Artifacts** â€” plots + summary markdown for the submission
7. **Backup** â€” push checkpoints and logs to a Hugging Face model repo so a session timeout doesn't kill the run
8. **Submission check** â€” final readiness validator

Each phase is independent â€” you can stop after the untrained eval, restart on a fresh runtime, and the resume logic will pick up where it left off.

## Runtime requirements

- CUDA GPU runtime â€” L4 (24 GB) or better recommended for Qwen3-4B
- HF token in `HF_TOKEN` env var (for model download + checkpoint backup)
- `WANDB_API_KEY` is optional (logs land in `grpo_wildfire/log.jsonl` regardless)

## 1. Install

In [ ]:
import os
import subprocess
import sys

REPO_URL = os.environ.get(
    "WILDFIRE_REPO_URL",
    "https://github.com/jhawaritvik/Scaler-hackathon.git",
)

if not os.path.isdir("wildfire-env"):
    subprocess.run(["git", "clone", REPO_URL, "wildfire-env"], check=True)
os.chdir("wildfire-env")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print("\nInstall complete. Working dir:", os.getcwd())

## 2. Preflight

`openenv validate` checks the manifest. `smoke_test.py` runs one tiny GRPO iteration end-to-end (~5 min on T4/L4) to catch OOM, NaN/inf, or dependency issues before committing to a full training run.

In [ ]:
!openenv validate .
!python smoke_test.py

## 3. Untrained baseline eval

Loads Qwen3-4B with no LoRA adapter and runs 5 held-out seeds per task. Output JSON has per-task means and per-seed scores. **Run this BEFORE training** â€” the trained-vs-untrained delta is the headline number for the submission.

Expected runtime: ~30-60 minutes on L4 (15 episodes Ã— ~2-4 min each).

In [ ]:
!mkdir -p submission_artifacts
!python eval_policy.py --untrained --output submission_artifacts/eval_untrained.json

## 4. Training

Multi-turn GRPO over the easy/medium/hard curriculum. Default config:

- 60 iterations, 4 trajectories per iter, 4 inner epochs
- Curriculum: easy 0-9, medium 10-24, hard 25-59
- 16 hand-curated training seeds per task (held-out from eval)

Checkpoints save to `grpo_wildfire/`:

- `latest/` â€” overwritten every iter (resume on disconnect)
- `best_adapter_<task>/` â€” written when grader score beats prior best for that task
- `adapter_iter{N:04d}/` â€” periodic snapshot every 10 iters

If the runtime disconnects, restart the cell â€” the resume logic picks up from `latest/`. **Estimated runtime: 18-30 hours on L4 for 60 iters.** If your budget is tight, reduce `total_iterations` in `train_grpo.py:Config` to 35-40.

In [ ]:
!python train_grpo.py

## 5. Trained eval

Re-runs the same five held-out seeds per task with the trained adapter. Auto-selects the latest periodic snapshot in `grpo_wildfire/`; pass `--adapter` to pick a specific checkpoint or `--adapter grpo_wildfire/best_adapter_hard` for the per-task best.

In [ ]:
!python eval_policy.py --output submission_artifacts/eval_trained.json

## 6. Artifacts

Generates the SVG/PNG reward and loss curves and a markdown training summary, all written under `submission_artifacts/`. The README references these files inline.

In [ ]:
!python plot_training_curves.py --log grpo_wildfire/log.jsonl --out-dir submission_artifacts
!python reward_audit.py --json-out reward_audit.json

## 6b. Capture demo replays

Captures one episode per task with the trained adapter and the heuristic baseline. The frames JSON streams through `/demo_replay` so you can record a screen capture for the submission video. After this cell finishes, open in a browser:

```
http://<your-host>:8000/viewer?replay=replays/trained_hard_9.json
```

In [ ]:
!mkdir -p replays
# Trained-LLM captures (one per task) — uses the per-task best adapter.
!python capture_replay.py --task easy   --seed 11 --policy llm --adapter grpo_wildfire/best_adapter_easy   --output replays/trained_easy_11.json
!python capture_replay.py --task medium --seed 1  --policy llm --adapter grpo_wildfire/best_adapter_medium --output replays/trained_medium_1.json
!python capture_replay.py --task hard   --seed 9  --policy llm --adapter grpo_wildfire/best_adapter_hard   --output replays/trained_hard_9.json

# Heuristic baseline captures on the same seeds (no GPU needed) — for side-by-side comparison.
!python capture_replay.py --task easy   --seed 11 --policy heuristic --output replays/heuristic_easy_11.json
!python capture_replay.py --task medium --seed 1  --policy heuristic --output replays/heuristic_medium_1.json
!python capture_replay.py --task hard   --seed 9  --policy heuristic --output replays/heuristic_hard_9.json

## 7. Backup checkpoints to a Hugging Face model repo

Highly recommended if you're on a hosted runtime (Colab, Kaggle, HF Spaces) where the disk is ephemeral. Set `HF_REPO_ID` to your own repo (e.g., `your-username/wildfire-grpo-checkpoints`) before running.

This uploads `grpo_wildfire/` (logs + checkpoints) so you can pull them back on a fresh runtime to continue training or run eval later.

In [ ]:
import os
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chunchunmaru-101/wildfire-grpo-checkpoints")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN env var to upload checkpoints.")

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path="grpo_wildfire",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="checkpoint upload",
    ignore_patterns=["*.pyc", "__pycache__/*"],
)
print(f"Uploaded grpo_wildfire/ â†’ https://huggingface.co/{HF_REPO_ID}")

## 8. Submission readiness check

Validates that all required artifacts are present and that README placeholders have been replaced with real links/results.

In [ ]:
!python submission_check.py --strict